# Fuzzy Feeder Paper-Ready Notebook

Notebook ini mengikuti acuan paper pada folder `paper` dan diselaraskan dengan firmware fuzzy baru di `feeder-platformio`.

Definisi sistem:

- Input 1: suhu air
- Input 2: biomassa ikan
- Output: jumlah pakan
- Inferensi: Mamdani dengan `MIN` dan `MAX`
- Defuzzifikasi: centroid

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

ASSET_DIR = Path('..') / 'paper_assets'
ASSET_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR

In [ ]:
def trimf(x, a, b, c):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    if b != a:
        left = (x >= a) & (x <= b)
        y[left] = (x[left] - a) / (b - a)
    if c != b:
        right = (x >= b) & (x <= c)
        y[right] = np.maximum(y[right], (c - x[right]) / (c - b))
    y[x == b] = 1.0
    return np.clip(y, 0.0, 1.0)


def interp_membership(universe, mf_values, crisp_value):
    return float(np.interp(crisp_value, universe, mf_values))


def centroid(universe, aggregated):
    denom = np.sum(aggregated)
    return 0.0 if denom <= 0 else float(np.sum(universe * aggregated) / denom)


def save_current_figure(filename):
    plt.savefig(ASSET_DIR / filename, dpi=300, bbox_inches='tight')

## Membership Function Sesuai Paper

In [ ]:
temp_universe = np.linspace(20, 40, 2001)
biomass_universe = np.linspace(0, 5500, 2201)
feed_universe = np.linspace(0, 150, 1501)

temp_mf = {
    'dingin': trimf(temp_universe, 20, 25, 30),
    'normal': trimf(temp_universe, 26, 30, 34),
    'panas': trimf(temp_universe, 28, 34, 38),
}

biomass_mf = {
    'sedikit': trimf(biomass_universe, 0, 1000, 2000),
    'sedang': trimf(biomass_universe, 1500, 2500, 3500),
    'banyak': trimf(biomass_universe, 2000, 4000, 5500),
}

feed_mf = {
    'sedikit': trimf(feed_universe, 0, 30, 60),
    'sedang': trimf(feed_universe, 40, 70, 100),
    'banyak': trimf(feed_universe, 80, 110, 150),
}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for label, mf in temp_mf.items():
    axes[0].plot(temp_universe, mf, label=label)
axes[0].set_title('Fungsi Keanggotaan Suhu')
axes[0].set_xlabel('Suhu (degC)')
axes[0].set_ylabel('Keanggotaan')
axes[0].legend()

for label, mf in biomass_mf.items():
    axes[1].plot(biomass_universe, mf, label=label)
axes[1].set_title('Fungsi Keanggotaan Biomassa')
axes[1].set_xlabel('Biomassa (g)')
axes[1].set_ylabel('Keanggotaan')
axes[1].legend()

for label, mf in feed_mf.items():
    axes[2].plot(feed_universe, mf, label=label)
axes[2].set_title('Fungsi Keanggotaan Output Pakan')
axes[2].set_xlabel('Pakan (g)')
axes[2].set_ylabel('Keanggotaan')
axes[2].legend()

plt.tight_layout()
save_current_figure('figure_membership_paper_model.png')
plt.show()

## Rule Base Sesuai Paper

In [ ]:
rules = [
    ('dingin', 'sedikit', 'sedikit'),
    ('dingin', 'sedang', 'sedikit'),
    ('dingin', 'banyak', 'sedang'),
    ('normal', 'sedikit', 'sedikit'),
    ('normal', 'sedang', 'sedang'),
    ('normal', 'banyak', 'banyak'),
    ('panas', 'sedikit', 'sedang'),
    ('panas', 'sedang', 'banyak'),
    ('panas', 'banyak', 'banyak'),
]

rule_df = pd.DataFrame(rules, columns=['suhu', 'biomassa', 'output'])
rule_df

In [ ]:
rule_matrix = rule_df.pivot(index='suhu', columns='biomassa', values='output')
rule_matrix

In [ ]:
rule_df.to_csv(ASSET_DIR / 'table_rule_base_paper.csv', index=False)
rule_matrix.to_csv(ASSET_DIR / 'table_rule_matrix_paper.csv')

## Fuzzifikasi, Inferensi, dan Defuzzifikasi Mamdani

In [ ]:
def fuzzy_inference(temp_value, biomass_value):
    temp_deg = {label: interp_membership(temp_universe, mf, temp_value) for label, mf in temp_mf.items()}
    biomass_deg = {label: interp_membership(biomass_universe, mf, biomass_value) for label, mf in biomass_mf.items()}
    output_alpha = {label: 0.0 for label in feed_mf}
    clipped_outputs = []
    rows = []

    for idx, (temp_label, biomass_label, output_label) in enumerate(rules, start=1):
        alpha = min(temp_deg[temp_label], biomass_deg[biomass_label])
        output_alpha[output_label] = max(output_alpha[output_label], alpha)
        clipped_outputs.append(np.minimum(alpha, feed_mf[output_label]))
        rows.append({
            'rule': idx,
            'suhu': temp_label,
            'biomassa': biomass_label,
            'output': output_label,
            'mu_suhu': temp_deg[temp_label],
            'mu_biomassa': biomass_deg[biomass_label],
            'alpha': alpha,
        })

    aggregated = np.max(np.vstack(clipped_outputs), axis=0)
    crisp_output = centroid(feed_universe, aggregated)
    return {
        'temp_deg': temp_deg,
        'biomass_deg': biomass_deg,
        'output_alpha': output_alpha,
        'firing_df': pd.DataFrame(rows),
        'aggregated': aggregated,
        'crisp_output': crisp_output,
    }

In [ ]:
temp_sample = 30.0
biomass_sample = 2000.0
result = fuzzy_inference(temp_sample, biomass_sample)
result['crisp_output']

In [ ]:
summary_df = pd.DataFrame([
    {
        'temp_input_degC': temp_sample,
        'biomass_input_g': biomass_sample,
        'defuzzified_feed_output_g': result['crisp_output'],
        'active_rule_count': int((result['firing_df']['alpha'] > 0).sum()),
    }
])
summary_df.to_csv(ASSET_DIR / 'table_example_case_paper.csv', index=False)
result['firing_df'].sort_values('alpha', ascending=False).to_csv(ASSET_DIR / 'table_firing_strength_paper.csv', index=False)
summary_df

In [ ]:
plt.figure(figsize=(10, 5))
for label, mf in feed_mf.items():
    plt.plot(feed_universe, mf, '--', alpha=0.3, label=f'{label} base')
plt.fill_between(feed_universe, result['aggregated'], color='tab:blue', alpha=0.4, label='Aggregated output')
plt.axvline(result['crisp_output'], color='red', linewidth=2, label=f"Centroid = {result['crisp_output']:.2f} g")
plt.title('Agregasi dan Defuzzifikasi Mamdani')
plt.xlabel('Pakan (g)')
plt.ylabel('Keanggotaan')
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
save_current_figure('figure_aggregation_paper.png')
plt.show()

In [ ]:
temp_grid = np.arange(22, 37, 1)
biomass_grid = np.arange(500, 5001, 250)
surface_rows = []
for t in temp_grid:
    for b in biomass_grid:
        out = fuzzy_inference(float(t), float(b))['crisp_output']
        surface_rows.append({'temp': t, 'biomass': b, 'feed_output': out})
surface_df = pd.DataFrame(surface_rows)
surface_df.to_csv(ASSET_DIR / 'table_surface_response_paper.csv', index=False)
surface_df.head()

In [ ]:
pivot_surface = surface_df.pivot(index='biomass', columns='temp', values='feed_output')
plt.figure(figsize=(10, 6))
plt.imshow(
    pivot_surface.values,
    aspect='auto',
    origin='lower',
    extent=[pivot_surface.columns.min(), pivot_surface.columns.max(), pivot_surface.index.min(), pivot_surface.index.max()],
    cmap='viridis'
)
plt.colorbar(label='Output Pakan (g)')
plt.title('Heatmap Surface Response Sistem Fuzzy')
plt.xlabel('Suhu (degC)')
plt.ylabel('Biomassa (g)')
save_current_figure('figure_surface_heatmap_paper.png')
plt.show()